In [2]:
for (year_to_process in 2018:2023) {
  library(data.table)
  library(readr)
  library(stringr)
  library(lubridate) # For handling timestamps

  # Load necessary parameter files
  source("~/drg-pipeline/data-cleaning/00a-parameters.r")
  to_sample <- TRUE
  suffix <- paste0(ifelse(exists("to_sample") && to_sample, paste0("_sampled_", sample_size_divisor, "_"), "_full_"))

  # Validate suffix
  valid_suffix_pattern <- "_full_|_sampled_\\d+_"
  if (!grepl(valid_suffix_pattern, suffix)) {
    stop("Invalid suffix: ", suffix, ". Expected '_full_' or '_sampled_<sample_size_divisor>_'")
  }

  # Read and filter filenames
  files <- list.files(
    path = "~/drg-pipeline/data-cleaning/data/chkpts/chkpt_12_mapping/",
    pattern = paste0("map_", year_to_process, suffix, ".*fork_\\d+_part_\\d+\\.rds"),
    full.names = TRUE
  )

  if (length(files) == 0) {
    message("No files found for the specified year and suffix: ", suffix)
    next
  }

  # Function to parse file names
  parse_filename <- function(filename) {
    pattern <- paste0("map_(\\d{4})", suffix, "(\\d{4}-\\d{2}-\\d{2} \\d{2}:\\d{2}:\\d{2}\\.\\d+)_fork_\\d+_part_\\d+\\.rds")
    matches <- str_match(filename, pattern)

    if (!is.na(matches[1])) {
      return(list(
        year = as.integer(matches[2]),
        timestamp = matches[3],
        filename = filename
      ))
    } else {
      return(NULL)
    }
  }

  parsed_files <- lapply(files, parse_filename)
  parsed_files <- Filter(Negate(is.null), parsed_files)

  if (length(parsed_files) == 0) {
    message("No valid parsed files found for year: ", year_to_process, " and suffix: ", suffix)
    next
  }

  # Select latest timestamp for the given year
  latest_timestamp <- max(sapply(parsed_files, `[[`, "timestamp"))
  latest_files <- Filter(function(x) x$timestamp == latest_timestamp, parsed_files)

  # Define output directories
  partial_output_folder <- paste0("~/drg-pipeline/data-cleaning/data/chkpts/chkpt_12_mapping/partial_mappings/", year_to_process, suffix, latest_timestamp, "/")
  final_output_folder <- paste0("~/drg-pipeline/data-cleaning/data/chkpts/chkpt_12_mapping/final_mappings/", year_to_process, suffix, latest_timestamp, "/")

  dir.create(final_output_folder, recursive = TRUE, showWarnings = FALSE)
  dir.create(partial_output_folder, recursive = TRUE, showWarnings = FALSE)

  # Read and combine mappings
  final_icd_dt <- rbindlist(lapply(latest_files, function(file_info) {
    mappings_list <- readRDS(file_info$filename)
    return(mappings_list$icd_mappings)
  }), use.names = TRUE, fill = TRUE)

  final_rvs_dt <- rbindlist(lapply(latest_files, function(file_info) {
    mappings_list <- readRDS(file_info$filename)
    return(mappings_list$rvs_mappings)
  }), use.names = TRUE, fill = TRUE)

  # Deduplicate while allowing multiple mappings per source code
  final_icd_dt <- unique(final_icd_dt)
  final_rvs_dt <- unique(final_rvs_dt)

  # Create final list and save as .rds with timestamp
  final_mappings_list <- list(final_icd_dt = final_icd_dt, final_rvs_dt = final_rvs_dt)
  output_rds <- paste0(final_output_folder, "final_map_", year_to_process, suffix, latest_timestamp, ".rds")
  saveRDS(final_mappings_list, output_rds)

  # Save debugging CSVs
  output_icd_csv <- paste0(final_output_folder, "icd_", year_to_process, suffix, latest_timestamp, ".csv")
  output_rvs_csv <- paste0(final_output_folder, "rvs_", year_to_process, suffix, latest_timestamp, ".csv")

  fwrite(final_icd_dt, output_icd_csv)
  fwrite(final_rvs_dt, output_rvs_csv)

  # Move partial .rds files to the corresponding folder (cut instead of copy)
  file.rename(files, file.path(partial_output_folder, basename(files)))

  message("Final mappings saved to: ", output_rds)
  message("ICD mappings saved to: ", output_icd_csv)
  message("RVS mappings saved to: ", output_rvs_csv)
  message("Partial mapping files moved to: ", partial_output_folder)
}


Parallelization: TRUE 


Final mappings saved to: ~/drg-pipeline/data-cleaning/data/chkpts/chkpt_12_mapping/final_mappings/2018_sampled_625_2025-02-24 00:37:34.447631/final_map_2018_sampled_625_2025-02-24 00:37:34.447631.rds

ICD mappings saved to: ~/drg-pipeline/data-cleaning/data/chkpts/chkpt_12_mapping/final_mappings/2018_sampled_625_2025-02-24 00:37:34.447631/icd_2018_sampled_625_2025-02-24 00:37:34.447631.csv

RVS mappings saved to: ~/drg-pipeline/data-cleaning/data/chkpts/chkpt_12_mapping/final_mappings/2018_sampled_625_2025-02-24 00:37:34.447631/rvs_2018_sampled_625_2025-02-24 00:37:34.447631.csv

Partial mapping files moved to: ~/drg-pipeline/data-cleaning/data/chkpts/chkpt_12_mapping/partial_mappings/2018_sampled_625_2025-02-24 00:37:34.447631/



Parallelization: TRUE 


Final mappings saved to: ~/drg-pipeline/data-cleaning/data/chkpts/chkpt_12_mapping/final_mappings/2019_sampled_625_2025-02-24 00:38:03.70624/final_map_2019_sampled_625_2025-02-24 00:38:03.70624.rds

ICD mappings saved to: ~/drg-pipeline/data-cleaning/data/chkpts/chkpt_12_mapping/final_mappings/2019_sampled_625_2025-02-24 00:38:03.70624/icd_2019_sampled_625_2025-02-24 00:38:03.70624.csv

RVS mappings saved to: ~/drg-pipeline/data-cleaning/data/chkpts/chkpt_12_mapping/final_mappings/2019_sampled_625_2025-02-24 00:38:03.70624/rvs_2019_sampled_625_2025-02-24 00:38:03.70624.csv

Partial mapping files moved to: ~/drg-pipeline/data-cleaning/data/chkpts/chkpt_12_mapping/partial_mappings/2019_sampled_625_2025-02-24 00:38:03.70624/



Parallelization: TRUE 


...

Final mappings saved to: ~/drg-pipeline/data-cleaning/data/chkpts/chkpt_12_mapping/final_mappings/2021_sampled_625_2025-02-24 00:39:02.406315/final_map_2021_sampled_625_2025-02-24 00:39:02.406315.rds

ICD mappings saved to: ~/drg-pipeline/data-cleaning/data/chkpts/chkpt_12_mapping/final_mappings/2021_sampled_625_2025-02-24 00:39:02.406315/icd_2021_sampled_625_2025-02-24 00:39:02.406315.csv

RVS mappings saved to: ~/drg-pipeline/data-cleaning/data/chkpts/chkpt_12_mapping/final_mappings/2021_sampled_625_2025-02-24 00:39:02.406315/rvs_2021_sampled_625_2025-02-24 00:39:02.406315.csv

Partial mapping files moved to: ~/drg-pipeline/data-cleaning/data/chkpts/chkpt_12_mapping/partial_mappings/2021_sampled_625_2025-02-24 00:39:02.406315/



Parallelization: TRUE 


Final mappings saved to: ~/drg-pipeline/data-cleaning/data/chkpts/chkpt_12_mapping/final_mappings/2022_sampled_625_2025-02-24 00:39:32.621367/final_map_2022_sampled_625_2025-02-24 00:39:32.621367.rds

ICD mappings saved to: ~/drg-pipeline/data-cleaning/data/chkpts/chkpt_12_mapping/final_mappings/2022_sampled_625_2025-02-24 00:39:32.621367/icd_2022_sampled_625_2025-02-24 00:39:32.621367.csv

RVS mappings saved to: ~/drg-pipeline/data-cleaning/data/chkpts/chkpt_12_mapping/final_mappings/2022_sampled_625_2025-02-24 00:39:32.621367/rvs_2022_sampled_625_2025-02-24 00:39:32.621367.csv

Partial mapping files moved to: ~/drg-pipeline/data-cleaning/data/chkpts/chkpt_12_mapping/partial_mappings/2022_sampled_625_2025-02-24 00:39:32.621367/



Parallelization: TRUE 


Final mappings saved to: ~/drg-pipeline/data-cleaning/data/chkpts/chkpt_12_mapping/final_mappings/2023_sampled_625_2025-02-24 00:40:00.982943/final_map_2023_sampled_625_2025-02-24 00:40:00.982943.rds

ICD mappings saved to: ~/drg-pipeline/data-cleaning/data/chkpts/chkpt_12_mapping/final_mappings/2023_sampled_625_2025-02-24 00:40:00.982943/icd_2023_sampled_625_2025-02-24 00:40:00.982943.csv

RVS mappings saved to: ~/drg-pipeline/data-cleaning/data/chkpts/chkpt_12_mapping/final_mappings/2023_sampled_625_2025-02-24 00:40:00.982943/rvs_2023_sampled_625_2025-02-24 00:40:00.982943.csv

Partial mapping files moved to: ~/drg-pipeline/data-cleaning/data/chkpts/chkpt_12_mapping/partial_mappings/2023_sampled_625_2025-02-24 00:40:00.982943/

